In [23]:
from pathlib import Path
import pandas as pd

datos = pd.read_csv("../data/processed/datos_procesados_2021.csv")

In [24]:
resumen_datos = pd.DataFrame({
    "Tipo de dato": datos.dtypes,
    "Valores faltantes": datos.isnull().sum(),
    "% Faltantes": (datos.isnull().sum() / len(datos) * 100).round(2)
})

resumen_datos

,Tipo de dato,Valores faltantes,% Faltantes
FechaFactura,str,0,0.00
IdRegistro,int64,0,0.00
Client,int64,0,0.00
Branch,int64,0,0.00
FechaApertura,str,0,0.00
FechaEntrega,str,3240,3.01
Factura,int64,0,0.00
Taller,str,0,0.00
TipoOrden,str,307,0.28
TipoPago,str,3658,3.39


In [25]:
datos["TipoOrden"] = datos["TipoOrden"].fillna("Sin Tipo Orden")
datos["TipoPago"] = datos["TipoPago"].fillna("Sin Tipo Pago")
datos["Color"] = datos["Color"].fillna("Desconocido")

In [26]:
resumen_datos = pd.DataFrame({
    "Tipo de dato": datos.dtypes,
    "Valores faltantes": datos.isnull().sum(),
    "% Faltantes": (datos.isnull().sum() / len(datos) * 100).round(2)
})

resumen_datos

,Tipo de dato,Valores faltantes,% Faltantes
FechaFactura,str,0,0.00
IdRegistro,int64,0,0.00
Client,int64,0,0.00
Branch,int64,0,0.00
FechaApertura,str,0,0.00
FechaEntrega,str,3240,3.01
Factura,int64,0,0.00
Taller,str,0,0.00
TipoOrden,str,0,0.00
TipoPago,str,0,0.00


In [27]:
datos["Taller"] = datos["Taller"].replace({
    "HOJALATERIA Y PINTURA INFINITI": "HOJALATERIA Y PINTURA",
    "SERVICIO INFINITI": "SERVICIO",
})

In [28]:

datos_modelo = datos.copy()

columnas_requeridas = [
    "FechaFactura", "Fuente", "IdRegistro", "Nombre", "Branch", "Estado",
    "Client", "Vin", "Ano", "Marca", "Color", "Taller", "TipoOrden",
    "TipoPago", "FechaEntrega", "FechaApertura", "NumeroOT", "Factura",
    "VentasNetas", "Venta$", "Costo$", "Utilidad$", "VentaMO$",
    "CostoMO$", "VentaTOT$", "CostoTOT$", "VentaPartes$",
    "CostoPartes$", "VentaTotal$", "CostoTotal$", "Odometro", "Dias",
    "TipoRegistro",
]
faltantes = [
    columna for columna in columnas_requeridas
    if columna not in datos_modelo.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias en datos: {faltantes}")

if datos_modelo["IdRegistro"].isna().any():
    raise ValueError("Hay registros sin IdRegistro.")

if datos_modelo["IdRegistro"].duplicated().any():
    raise ValueError("IdRegistro está duplicado; revisa el dataframe datos.")

if datos_modelo["Vin"].isna().any():
    raise ValueError("Hay valores nulos en Vin, que es clave de DIM_VEHICULO.")


raiz_proyecto = next(
    (
        carpeta
        for carpeta in (Path.cwd(), *Path.cwd().parents)
        if (carpeta / ".git").exists()
    ),
    Path.cwd(),
)
carpeta_dashboard = raiz_proyecto / "Dashboard"
carpeta_dashboard.mkdir(parents=True, exist_ok=True)


def obtener_claves(tabla, dimension, columnas, nombre_clave):
    if dimension.duplicated(columnas).any():
        raise ValueError(
            f"La dimensión tiene claves duplicadas en {columnas}."
        )

    indice_dimension = pd.MultiIndex.from_frame(dimension[columnas])
    indice_datos = pd.MultiIndex.from_frame(tabla[columnas])
    posiciones = indice_dimension.get_indexer(indice_datos)

    if (posiciones < 0).any():
        raise ValueError(f"No se pudo asignar la clave {nombre_clave}.")

    return dimension[nombre_clave].to_numpy()[posiciones]



datos_modelo["_Fecha"] = pd.to_datetime(
    datos_modelo["FechaFactura"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
).dt.normalize()

dim_fecha = (
    datos_modelo[["_Fecha", "Fuente"]]
    .drop_duplicates()
    .sort_values(["_Fecha", "Fuente"], na_position="last")
    .reset_index(drop=True)
)

dim_fecha["Anio"] = dim_fecha["_Fecha"].dt.year.astype("Int64")
dim_fecha["Mes"] = dim_fecha["_Fecha"].dt.month.astype("Int64")

nombres_meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre",
}
dim_fecha["NombreMes"] = (
    dim_fecha["Mes"].map(nombres_meses).fillna("Sin definir")
)
dim_fecha.insert(0, "FechaFactura", range(1, len(dim_fecha) + 1))

clave_fecha = obtener_claves(
    datos_modelo,
    dim_fecha,
    ["_Fecha", "Fuente"],
    "FechaFactura",
)

dim_fecha = dim_fecha.rename(columns={"_Fecha": "Fecha"})
dim_fecha = dim_fecha[
    ["FechaFactura", "Fecha", "Anio", "Mes", "NombreMes", "Fuente"]
]



datos_modelo["NombreSucursal"] = datos_modelo["Nombre"]
atributos_sucursal = ["NombreSucursal", "Branch", "Estado", "Client"]

dim_sucursal = (
    datos_modelo[atributos_sucursal]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_sucursal.insert(0, "Nombre", range(1, len(dim_sucursal) + 1))

clave_sucursal = obtener_claves(
    datos_modelo,
    dim_sucursal,
    atributos_sucursal,
    "Nombre",
)


variaciones_vin = (
    datos_modelo.groupby("Vin", sort=False)[["Ano", "Marca", "Color"]]
    .nunique(dropna=True)
)
vin_inconsistentes = (variaciones_vin > 1).any(axis=1).sum()
print(f"VIN con atributos inconsistentes: {vin_inconsistentes:,}")

vehiculos_base = (
    datos_modelo[["Vin", "Ano", "Marca", "Color"]]
    .reset_index(drop=True)
    .copy()
)
vehiculos_base["_orden"] = range(len(vehiculos_base))


def valor_mas_frecuente_por_vin(columna):
    conteos = (
        vehiculos_base
        .groupby(["Vin", columna], dropna=False, sort=False)
        .agg(
            frecuencia=("_orden", "size"),
            primer_registro=("_orden", "min"),
        )
        .reset_index()
    )

    seleccionados = (
        conteos
        .sort_values(
            ["Vin", "frecuencia", "primer_registro"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates("Vin")
        .set_index("Vin")[columna]
    )

    return seleccionados


dim_vehiculo = pd.concat(
    [
        valor_mas_frecuente_por_vin("Ano").rename("Ano"),
        valor_mas_frecuente_por_vin("Marca").rename("Marca"),
        valor_mas_frecuente_por_vin("Color").rename("Color"),
    ],
    axis=1,
).reset_index()

atributos_servicio = ["Taller", "TipoOrden", "TipoPago"]
dim_servicio = (
    datos_modelo[atributos_servicio]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_servicio.insert(0, "sk_servicio", range(1, len(dim_servicio) + 1))

clave_servicio = obtener_claves(
    datos_modelo,
    dim_servicio,
    atributos_servicio,
    "sk_servicio",
)


codigos_tipo_registro = {
    "registro de venta": 1,
    "registro de cancelacion": 2,
    "registro de cancelación": 2,
    "registro cancelado": 3,
}

tipo_registro = (
    datos_modelo["TipoRegistro"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

tipos_no_reconocidos = (
    set(tipo_registro.dropna().unique())
    - set(codigos_tipo_registro)
)
if tipos_no_reconocidos:
    raise ValueError(
        f"Hay valores de TipoRegistro sin código: {tipos_no_reconocidos}"
    )

codigo_tipo_registro = (
    tipo_registro.map(codigos_tipo_registro)
    .fillna(0)
    .astype("int64")
)

fact_ordenes = pd.DataFrame(index=datos_modelo.index)
fact_ordenes["FechaFactura"] = clave_fecha
fact_ordenes["Nombre"] = clave_sucursal
fact_ordenes["Vin"] = datos_modelo["Vin"].to_numpy()
fact_ordenes["sk_servicio"] = clave_servicio
fact_ordenes["IdRegistro"] = datos_modelo["IdRegistro"].to_numpy()

fact_ordenes["FechaEntrega"] = pd.to_datetime(
    datos_modelo["FechaEntrega"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
)
fact_ordenes["FechaApertura"] = pd.to_datetime(
    datos_modelo["FechaApertura"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
)

fact_ordenes["NumeroOT"] = datos_modelo["NumeroOT"].to_numpy()
fact_ordenes["Factura"] = datos_modelo["Factura"].to_numpy()
fact_ordenes["VentasNetas"] = datos_modelo["VentasNetas"].to_numpy()

medidas = {
    "Venta$": "Venta",
    "Costo$": "Costo",
    "Utilidad$": "Utilidad",
    "VentaMO$": "VentaMO",
    "CostoMO$": "CostoMO",
    "VentaTOT$": "VentaTOT",
    "CostoTOT$": "CostoTOT",
    "VentaPartes$": "VentaPartes",
    "CostoPartes$": "CostoPartes",
    "VentaTotal$": "VentaTotal",
    "CostoTotal$": "CostoTotal",
}
for columna_origen, columna_destino in medidas.items():
    fact_ordenes[columna_destino] = datos_modelo[columna_origen].to_numpy()

fact_ordenes["Odometro"] = datos_modelo["Odometro"].to_numpy()
fact_ordenes["Dias"] = datos_modelo["Dias"].to_numpy()
fact_ordenes["TipoRegistro"] = codigo_tipo_registro.to_numpy()

fact_ordenes = fact_ordenes[
    [
        "FechaFactura", "Nombre", "Vin", "sk_servicio",
        "IdRegistro", "FechaEntrega", "FechaApertura",
        "NumeroOT", "Factura", "VentasNetas", "Venta", "Costo",
        "Utilidad", "VentaMO", "CostoMO", "VentaTOT", "CostoTOT",
        "VentaPartes", "CostoPartes", "VentaTotal", "CostoTotal",
        "Odometro", "Dias", "TipoRegistro",
    ]
]

for columna_fk, dimension, columna_pk in [
    ("FechaFactura", dim_fecha, "FechaFactura"),
    ("Nombre", dim_sucursal, "Nombre"),
    ("Vin", dim_vehiculo, "Vin"),
    ("sk_servicio", dim_servicio, "sk_servicio"),
]:
    if not fact_ordenes[columna_fk].isin(dimension[columna_pk]).all():
        raise ValueError(f"Hay claves foráneas inválidas en {columna_fk}.")

if len(fact_ordenes) != len(datos_modelo):
    raise ValueError("FACT_ORDENES no conserva todas las filas de datos.")

tablas = {
    "DIM_FECHA": dim_fecha,
    "DIM_SUCURSAL": dim_sucursal[
        ["Nombre", "Branch", "Estado", "Client", "NombreSucursal"]
    ],
    "DIM_VEHICULO": dim_vehiculo[
        ["Vin", "Ano", "Marca", "Color"]
    ],
    "DIM_SERVICIO": dim_servicio[
        ["sk_servicio", "Taller", "TipoOrden", "TipoPago"]
    ],
    "FACT_ORDENES": fact_ordenes,
}

for nombre, tabla in tablas.items():
    destino = carpeta_dashboard / f"{nombre}.csv"
    tabla.to_csv(destino, index=False, encoding="utf-8-sig")
    print(f"Guardado: {destino} ({len(tabla):,} filas)")

VIN con atributos inconsistentes: 886
Guardado: c:\Users\Administrador\Desktop\Examen BI\Examen-BI\Dashboard\DIM_FECHA.csv (174 filas)
Guardado: c:\Users\Administrador\Desktop\Examen BI\Examen-BI\Dashboard\DIM_SUCURSAL.csv (30 filas)
Guardado: c:\Users\Administrador\Desktop\Examen BI\Examen-BI\Dashboard\DIM_VEHICULO.csv (69,838 filas)
Guardado: c:\Users\Administrador\Desktop\Examen BI\Examen-BI\Dashboard\DIM_SERVICIO.csv (16 filas)
Guardado: c:\Users\Administrador\Desktop\Examen BI\Examen-BI\Dashboard\FACT_ORDENES.csv (107,787 filas)
